In [1]:
import sqlite3
import pandas as pd

# Load the cleaned data
data_path = r'C:\Users\sunka\Documents\victoria-road-crash-analysis\data\processed'

accident_clean = pd.read_csv(data_path + r'\accident_clean.csv')
fatal_analysis = pd.read_csv(data_path + r'\fatal_analysis.csv')
speed_summary  = pd.read_csv(data_path + r'\speed_zone_summary.csv')

# Create a SQLite database in memory
conn = sqlite3.connect(':memory:')

# Load the dataframes into the database as SQL tables
accident_clean.to_sql('accidents', conn, index=False, if_exists='replace')
fatal_analysis.to_sql('fatalities', conn, index=False, if_exists='replace')
speed_summary.to_sql('speed_zones', conn, index=False, if_exists='replace')

print("✅ SQLite database created!")
print("Tables available:")
print("- accidents")
print("- fatalities")
print("- speed_zones")

✅ SQLite database created!
Tables available:
- accidents
- fatalities
- speed_zones


In [2]:
# SQL QUERY 1: How have crashes and fatalities changed each year?
query1 = """
SELECT 
    YEAR,
    COUNT(*) AS total_crashes,
    SUM(NO_PERSONS_KILLED) AS total_fatalities,
    ROUND(SUM(NO_PERSONS_KILLED) * 100.0 / COUNT(*), 2) AS fatality_rate_pct
FROM accidents
GROUP BY YEAR
ORDER BY YEAR
"""

result1 = pd.read_sql_query(query1, conn)
print("=== CRASHES AND FATALITIES BY YEAR ===")
print(result1.to_string(index=False))

=== CRASHES AND FATALITIES BY YEAR ===
 YEAR  total_crashes  total_fatalities  fatality_rate_pct
 2012          13113               280               2.14
 2013          13172               239               1.81
 2014          13543               248               1.83
 2015          14595               249               1.71
 2016          14686               287               1.95
 2017          12513               253               2.02
 2018          11931               208               1.74
 2019          13281               259               1.95
 2020          11162               210               1.88
 2021          12914               228               1.77
 2022          13798               239               1.73
 2023          14277               289               2.02
 2024          13917               275               1.98
 2025          12495               280               2.24


In [3]:
# SQL QUERY 2: Which hours are most deadly?
query2 = """
SELECT 
    CAST(HOUR AS INTEGER) AS hour_of_day,
    COUNT(*) AS fatalities,
    CASE 
        WHEN CAST(HOUR AS INTEGER) BETWEEN 6 AND 11 THEN 'Morning'
        WHEN CAST(HOUR AS INTEGER) BETWEEN 12 AND 17 THEN 'Afternoon'
        WHEN CAST(HOUR AS INTEGER) BETWEEN 18 AND 22 THEN 'Evening'
        ELSE 'Night/Early Morning'
    END AS time_period
FROM fatalities
GROUP BY CAST(HOUR AS INTEGER)
ORDER BY fatalities DESC
LIMIT 10
"""

result2 = pd.read_sql_query(query2, conn)
print("=== TOP 10 DEADLIEST HOURS ===")
print(result2.to_string(index=False))

=== TOP 10 DEADLIEST HOURS ===
 hour_of_day  fatalities time_period
          15         255   Afternoon
          16         251   Afternoon
          14         220   Afternoon
          13         199   Afternoon
          12         192   Afternoon
          17         191   Afternoon
          11         189     Morning
          18         181     Evening
           6         156     Morning
          10         150     Morning


In [4]:
# SQL QUERY 3: Who is dying? Age group and sex breakdown
query3 = """
SELECT 
    AGE_GROUP,
    SEX,
    COUNT(*) AS fatalities,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_total
FROM fatalities
WHERE SEX IN ('M', 'F')
AND AGE_GROUP != 'Unknown'
GROUP BY AGE_GROUP, SEX
ORDER BY fatalities DESC
LIMIT 15
"""

result3 = pd.read_sql_query(query3, conn)
print("=== FATALITIES BY AGE GROUP AND SEX ===")
print(result3.to_string(index=False))

=== FATALITIES BY AGE GROUP AND SEX ===
AGE_GROUP SEX  fatalities  pct_of_total
    30-39   M         431          12.0
      70+   M         417          11.6
    40-49   M         350           9.7
    50-59   M         332           9.2
      70+   F         271           7.5
    18-21   M         252           7.0
    26-29   M         215           6.0
    22-25   M         212           5.9
    60-64   M         140           3.9
    65-69   M         128           3.6
    50-59   F         113           3.1
    30-39   F         103           2.9
    40-49   F          95           2.6
    18-21   F          83           2.3
    60-64   F          73           2.0


In [5]:
# SQL QUERY 4: Which speed zones are most deadly?
query4 = """
SELECT 
    SPEED_ZONE,
    TOTAL_CRASHES,
    FATALITIES,
    FATAL_RATE_PCT,
    CASE
        WHEN SPEED_ZONE <= 60 THEN 'Urban'
        WHEN SPEED_ZONE <= 80 THEN 'Suburban/Mixed'
        ELSE 'Regional/Highway'
    END AS road_type
FROM speed_zones
ORDER BY FATAL_RATE_PCT DESC
"""

result4 = pd.read_sql_query(query4, conn)
print("=== FATAL RATE BY SPEED ZONE ===")
print(result4.to_string(index=False))

=== FATAL RATE BY SPEED ZONE ===
 SPEED_ZONE  TOTAL_CRASHES  FATALITIES  FATAL_RATE_PCT        road_type
        110           2087         121             5.8 Regional/Highway
        100          28218        1497             5.3 Regional/Highway
         75             20           1             5.0   Suburban/Mixed
         90            505          22             4.4 Regional/Highway
         80          30377         616             2.0   Suburban/Mixed
         70          12540         198             1.6   Suburban/Mixed
         60          65305         695             1.1            Urban
         50          32987         307             0.9            Urban
         40          12937          74             0.6            Urban
         30            421           2             0.5            Urban


In [6]:
# SQL QUERY 5: Which day of week has most fatalities?
query5 = """
SELECT 
    DAY_OF_WEEK,
    COUNT(*) AS fatalities,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_total,
    CASE
        WHEN DAY_OF_WEEK IN ('Saturday','Sunday') THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type
FROM fatalities
GROUP BY DAY_OF_WEEK
ORDER BY fatalities DESC
"""

result5 = pd.read_sql_query(query5, conn)
print("=== FATALITIES BY DAY OF WEEK ===")
print(result5.to_string(index=False))

=== FATALITIES BY DAY OF WEEK ===
DAY_OF_WEEK  fatalities  pct_of_total day_type
   Saturday         565          15.7  Weekend
     Friday         554          15.4  Weekday
     Sunday         539          15.0  Weekend
   Thursday         518          14.4  Weekday
  Wednesday         466          13.0  Weekday
     Monday         464          12.9  Weekday
    Tuesday         427          11.9  Weekday
        NaN          62           1.7  Weekday


In [7]:
# Save all SQL results as CSV files for Tableau
sql_path = r'C:\Users\sunka\Documents\victoria-road-crash-analysis\sql'

result1.to_csv(sql_path + r'\query1_yearly_trend.csv', index=False)
result2.to_csv(sql_path + r'\query2_hourly_fatalities.csv', index=False)
result3.to_csv(sql_path + r'\query3_age_sex_fatalities.csv', index=False)
result4.to_csv(sql_path + r'\query4_speed_zone_fatal_rate.csv', index=False)
result5.to_csv(sql_path + r'\query5_day_of_week.csv', index=False)

print("✅ All SQL results saved!")
print("\nFiles saved to sql/ folder:")
print("- query1_yearly_trend.csv")
print("- query2_hourly_fatalities.csv")
print("- query3_age_sex_fatalities.csv")
print("- query4_speed_zone_fatal_rate.csv")
print("- query5_day_of_week.csv")

✅ All SQL results saved!

Files saved to sql/ folder:
- query1_yearly_trend.csv
- query2_hourly_fatalities.csv
- query3_age_sex_fatalities.csv
- query4_speed_zone_fatal_rate.csv
- query5_day_of_week.csv
